# Proyecto integrador — Pipeline predictivo y despliegue ético (Unidad 4)

Este notebook integra evidencia predictiva y criterios éticos para decisiones financieras:

1. Orquestación de datos (ingesta, limpieza, split train/validation/test).
2. Entrenamiento y validación (clasificación de default + regresión de pérdida esperada).
3. Monitoreo (drift de datos y caída de desempeño).
4. Retraining con reglas auditables.
5. Umbrales, sensibilidad e impacto económico por escenarios.

> Reproducible: usa semilla fija y funciones en `src/pipeline_financiero.py`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, roc_auc_score

pd.set_option("display.max_columns", None)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from pipeline_financiero import (
    FEATURE_COLUMNS,
    decidir_retraining,
    entrenar_modelos,
    evaluar_modelos,
    generar_datos_sinteticos,
    monitorear_drift,
    preparar_datos,
)

RANDOM_STATE = 42

In [ ]:
# 1) Ingesta + limpieza + split
data = generar_datos_sinteticos(n_muestras=6000, semilla=RANDOM_STATE)
split = preparar_datos(data, semilla=RANDOM_STATE)

print("Tamaño total:", data.shape)
print("Train:", split.x_train.shape, "Validation:", split.x_val.shape, "Test:", split.x_test.shape)
print("Tasa de default global:", round(data["default"].mean(), 4))

data.head(3)

In [ ]:
# 2) Entrenamiento y validación (2 modelos)
modelos = entrenar_modelos(split)
metricas_base = evaluar_modelos(split, modelos)

pd.DataFrame([metricas_base], index=["baseline_validacion"]).round(4)

In [ ]:
# 3) Monitoreo: simulación de datos del mes actual con drift
rng = np.random.default_rng(2026)
data_mes_actual = split.x_test.copy()

data_mes_actual["ratio_deuda_ingreso"] = (data_mes_actual["ratio_deuda_ingreso"] + 0.08).clip(0, 1)
data_mes_actual["utilizacion_credito"] = (data_mes_actual["utilizacion_credito"] + 0.05).clip(0, 1)
data_mes_actual["caida_ingresos_12m"] = (data_mes_actual["caida_ingresos_12m"] + rng.normal(0.03, 0.02, len(data_mes_actual))).clip(-0.2, 0.6)

drift_df = monitorear_drift(split.x_train, data_mes_actual)
drift_df.round(4)

In [ ]:
# 4) Degradación de performance en la ventana actual
clf = modelos["clasificacion_default"]
reg = modelos["regresion_perdida"]

y_prob_actual = clf.predict_proba(data_mes_actual)[:, 1]
y_reg_actual = reg.predict(data_mes_actual)

metricas_actuales = {
    "roc_auc": float(roc_auc_score(split.y_test_clf, y_prob_actual)),
    "mae": float(mean_absolute_error(split.y_test_reg, y_reg_actual)),
}

comparativo_metricas = pd.DataFrame(
    [
        {"roc_auc": metricas_base["roc_auc"], "mae": metricas_base["mae"]},
        metricas_actuales,
    ],
    index=["baseline_validacion", "ventana_actual_test"],
)
comparativo_metricas.round(4)

In [ ]:
# 5) Trigger de retraining (tiempo + drift + desempeño)
decision_retrain = decidir_retraining(
    baseline_metrics={"roc_auc": metricas_base["roc_auc"], "mae": metricas_base["mae"]},
    current_metrics=metricas_actuales,
    drift_df=drift_df,
    fecha_ultimo_entrenamiento="2026-01-15",
    fecha_actual="2026-05-10",
)
decision_retrain

In [ ]:
# 6) Umbrales y sensibilidad (criterios de activación)
criterios = [
    {"criterio": "C1", "regla": "Activar alerta si PD > 0.70", "activo": float((y_prob_actual > 0.70).mean())},
    {
        "criterio": "C2",
        "regla": "Activar alerta si caída de ingresos > 15%",
        "activo": float((data_mes_actual["caida_ingresos_12m"] > 0.15).mean()),
    },
    {
        "criterio": "C3",
        "regla": "Activar alerta si PSI promedio > 0.20",
        "activo": float(drift_df["psi"].mean() > 0.20),
    },
]
criterios_df = pd.DataFrame(criterios)

sensibilidad = []
for umbral in [0.50, 0.60, 0.70, 0.80]:
    aprobados = y_prob_actual < umbral
    sensibilidad.append(
        {
            "umbral_pd": umbral,
            "tasa_aprobacion": float(aprobados.mean()),
            "default_en_aprobados": float(split.y_test_clf[aprobados].mean()),
            "default_en_rechazados": float(split.y_test_clf[~aprobados].mean()) if (~aprobados).sum() > 0 else 0.0,
        }
    )

sensibilidad_df = pd.DataFrame(sensibilidad)
display(criterios_df)
sensibilidad_df.round(4)

In [ ]:
# 7) Impacto económico por escenarios (optimista/base/pesimista)
portfolio = data_mes_actual.copy()
portfolio["pd"] = y_prob_actual
portfolio["ead_k"] = (portfolio["saldo_promedio"] / 1000).clip(lower=3)
portfolio["lgd"] = (0.30 + 0.55 * portfolio["ratio_deuda_ingreso"]).clip(0.20, 0.85)

umbral_operativo = 0.70
aprobados = portfolio["pd"] < umbral_operativo

escenarios = {
    "optimista": {"mult_pd": 0.85, "margen_k": 0.52},
    "base": {"mult_pd": 1.00, "margen_k": 0.45},
    "pesimista": {"mult_pd": 1.30, "margen_k": 0.38},
}

rows = []
for nombre, params in escenarios.items():
    pd_adj = (portfolio["pd"] * params["mult_pd"]).clip(0, 0.99)
    perdida_sin_politica = float((pd_adj * portfolio["ead_k"] * portfolio["lgd"]).sum())
    perdida_con_politica = float((pd_adj[aprobados] * portfolio.loc[aprobados, "ead_k"] * portfolio.loc[aprobados, "lgd"]).sum())
    ingresos_k = float(aprobados.sum() * params["margen_k"])
    beneficio_neto_k = ingresos_k - perdida_con_politica
    ahorro_vs_sin_politica_k = perdida_sin_politica - perdida_con_politica
    rows.append(
        {
            "escenario": nombre,
            "perdida_sin_politica_k": perdida_sin_politica,
            "perdida_con_politica_k": perdida_con_politica,
            "ahorro_vs_sin_politica_k": ahorro_vs_sin_politica_k,
            "ingresos_k": ingresos_k,
            "beneficio_neto_k": beneficio_neto_k,
        }
    )

escenarios_df = pd.DataFrame(rows)
escenarios_df.round(2)

In [ ]:
# 8) Evidencia de trazabilidad para auditoría: persistencia de artefactos
artifacts_dir = ROOT / "artifacts"
artifacts_dir.mkdir(exist_ok=True)

drift_df.to_csv(artifacts_dir / "drift_indicadores.csv", index=False)
sensibilidad_df.to_csv(artifacts_dir / "sensibilidad_umbrales.csv", index=False)
escenarios_df.to_csv(artifacts_dir / "escenarios_impacto_economico.csv", index=False)
pd.DataFrame([decision_retrain]).to_json(artifacts_dir / "decision_retraining.json", orient="records", indent=2)
comparativo_metricas.to_csv(artifacts_dir / "comparativo_metricas.csv")

print("Artefactos guardados en:", artifacts_dir)
sorted([p.name for p in artifacts_dir.iterdir()])

## Pregunta de examen (4.1): trazabilidad para comité regulador

Para garantizar trazabilidad auditada:

1. Versionar código, parámetros y datos de entrada (hash de datasets por corte).
2. Registrar métricas baseline y por ventana en bitácora inmutable.
3. Documentar reglas de decisión (umbrales y triggers de retraining) en forma explícita.
4. Persistir artefactos de monitoreo (drift, caída de ROC AUC/MAE, decisión final).
5. Definir responsables (owner técnico y owner de riesgo) con sello de aprobación.